## Analisi della densità polifonica

Questo notebook analizza la densità polifonica di una composizione: quante e quali voci sono di volta in volta attive.

### Importare il codice

Importa le librerie necessarie e definisce `download_html`, che consente di salvare ogni grafico come file HTML interattivo.

In [ ]:
from itertools import cycle
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML
import base64
from crim_intervals import importScore
import plotly.graph_objects as go

def download_html(fig, filename='chart.html'):
    b64 = base64.b64encode(fig.to_html(include_plotlyjs='cdn').encode()).decode()
    display(HTML(
        f'<a href="data:text/html;base64,{b64}" download="{filename}" '
        f'style="display:inline-block;margin-top:6px;padding:4px 14px;background:#4472C4;'
        f'color:#fff;border-radius:4px;text-decoration:none;font-size:12px;">&#8595; {filename}</a>'
    ))

print('Ok')

#### Caricamento della composizione ####

Caricamento di un file MusicXML dalla cartella locale `Music_Files/`. Modificare il valore di `filename` per cambiare composizione:

- `'Marenzio_o_voi.xml'` — *O voi che sospirate*, Luca Marenzio
- `'Monteverdi_o_rossignuol.xml'` — *O rossignuol*, Claudio Monteverdi

Se il caricamento è andato a buon fine, `print(piece.metadata)` restituisce titolo e compositore.

In [ ]:
filename = 'Monteverdi_o_rossignuol.xml'  # ← cambia qui per caricare un altro file
url   = f'Music_Files/{filename}'
piece = importScore(url)
print(piece.metadata)

In alternativa, è possibile caricare un file da un URL pubblico. Eseguire questa cella *oppure* la precedente, non entrambe.

In [ ]:
url = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(url)
print(piece.metadata)

#### Note e durate

`piece.notes()` restituisce le note e le pause della composizione in forma di tabella. Ogni colonna corrisponde a una voce; l'indice di riga è l'*offset*, ossia la posizione temporale espressa in semiminime a partire da 0.

In [ ]:
n = piece.notes()
n

`piece.durations()` restituisce la durata di ogni nota, nella stessa forma tabellare e con lo stesso indice.

In [ ]:
d = piece.durations()
d

#### Numero di voci attive

`piece.soundingCount()` restituisce, per ogni evento, il numero di voci di volta in volta attive (*sounding*).

In [ ]:
sound = piece.soundingCount()
sound

La stessa informazione può essere visualizzata rapidamente come grafico a barre tramite `.plot.bar()` di pandas.

In [ ]:
sound_plot = sound.plot.bar(figsize=(20, 4), width=1.0, xticks=[])
print('Doppio click sul grafico per ingrandire/rimpicciolire')
sound_plot


#### Cadenze

`piece.cadences()` individua le cadenze sulla base di pattern contrappuntistici predefiniti. Per ogni cadenza vengono indicati la tipologia, la sonorità di approdo e altre informazioni.

In [ ]:
cad = piece.cadences()
cad

`piece.cadenceProgressPlot()` visualizza la successione delle cadenze: sull'asse x il *progress* (posizione proporzionale da 0 a 1), sull'asse y la sonorità di approdo. Con `includeType=True` i marcatori variano forma a seconda del tipo di cadenza.

In [ ]:
piece.cadenceProgressPlot(includeType=True)

In [ ]:
# piece.linkExamples() genera link agli esempi musicali in EMA Viewer.
# Funziona solo con file caricati da URL pubblici (celle 6-7).
cad = piece.cadences()
link_cad = piece.linkExamples(df=cad, piece_url=url)

#### Testo

`piece.lyrics()` restituisce le sillabe associate a ciascuna nota, nella stessa forma tabellare degli altri dati.

In [ ]:
lyr = piece.lyrics()
lyr

### Combinare i dati

I dati raccolti vengono qui uniti in un'unica tabella, che costituisce la base per i grafici nella sezione successiva. Le informazioni sulle cadenze vengono aggiunte tramite un *merge* su battuta (*measure*) e pulsazione (*beat*).

In [ ]:
n     = piece.notes()
d     = piece.durations()
sound = piece.soundingCount()
lyr   = piece.lyrics().rename(columns=lambda v: f'Lyric_{v}')

output = piece.detailIndex(
    df=pd.concat([n, d, sound, lyr], axis=1),
    progress=True
).reset_index()

output['Composer'] = piece.metadata['composer']
output['Title']    = piece.metadata['title']
output['path']     = Path(url).name

first_cols = ['Measure', 'Beat', 'Composer', 'Title', 'path']
output = output[first_cols + [c for c in output.columns if c not in first_cols]]

cad = piece.cadences()
output = output.merge(
    cad[['Measure', 'Beat', 'CadType', 'Tone', 'CVFs']],
    on=['Measure', 'Beat'],
    how='left'
)

# normalizza progress sulla fine effettiva del brano
# (ultimo attacco + durata massima), così le ultime note non vengono tagliate
total_offsets = (d.index + d.max(axis=1)).max()
output['Progress'] *= d.index.max() / total_offsets

output

## Grafici

### Attività delle voci

Ogni riga corrisponde a una voce; ogni 'mattone' rappresenta una nota, posizionata in base alla sua posizione nella composizione e della dimensione corrispondente alla sua durata. Le linee rosse tratteggiate segnalano le cadenze rilevate da `piece.cadences()`; il righello in basso indica i numeri di battuta. Passando il cursore sopra ciascun mattoncino sarà possibile leggere l'altezza, la posizione e la sillaba associata a ciascuna nota.

In [ ]:
# voice columns: each voice appears twice in output (notes, then durations)
all_cols = list(output.columns)
voices   = n.columns.tolist()
note_idx = {v: all_cols.index(v)                for v in voices}
dur_idx  = {v: all_cols.index(v, note_idx[v]+1) for v in voices}

# measure ruler — reused in all three chart cells
measures       = output.groupby('Measure')['Progress'].min()
measure_lines  = [
    dict(type='line', x0=p, x1=p, y0=0, y1=1,
         xref='x', yref='paper', line=dict(color='#ddd', width=0.5))
    for p in measures.values
]
measure_labels = [
    dict(x=p, y=0, xref='x', yref='paper', text=str(int(m)),
         showarrow=False, yanchor='top', xanchor='center', font=dict(size=8))
    for m, p in measures.items() if m == 1 or m % 5 == 0
]

file_stem = url.split('/')[-1].rsplit('.', 1)[0]

# cadence markers — reused in all three chart cells
CAD_ABBR = {'Clausula Vera': 'CV', 'Authentic': 'Auth.', 'Evaded': 'Evad.', 'Abandoned': 'Aband.'}

def shorten_cad(name):
    for word, abbr in CAD_ABBR.items():
        name = name.replace(word, abbr)
    return name.strip()

cad_rows  = output.dropna(subset=['CadType'])
cad_lines = [
    dict(type='line', x0=row['Progress'], x1=row['Progress'], y0=0, y1=1,
         xref='x', yref='paper', line=dict(color='#c00', width=1, dash='dot'))
    for _, row in cad_rows.iterrows()
]
cad_labels = [
    dict(x=row['Progress'], y=1, xref='x', yref='paper',
         text=shorten_cad(row['CadType']) + (f"<br><b>{row['Tone']}</b>" if pd.notna(row['Tone']) else ''),
         showarrow=False, yanchor='bottom', xanchor='center',
         font=dict(size=7, color='#c00'))
    for _, row in cad_rows.iterrows()
]

palette = ['#4472C4','#ED7D31','#70AD47','#FFC000','#7030A0',
           '#FF6B6B','#00B0F0','#92D050','#FF00FF','#00B050']

def add_voice_traces(fig, row=None, col=None):
    for voice, color in zip(voices, cycle(palette)):
        notes   = output.iloc[:, note_idx[voice]]
        durs    = output.iloc[:, dur_idx[voice]]
        active  = notes.notna() & (notes != 'Rest')
        lyr_col = f'Lyric_{voice}'
        lyrics  = (output[lyr_col][active].fillna('').astype(str)
                   if lyr_col in output.columns
                   else pd.Series('', index=notes[active].index))
        fig.add_trace(go.Bar(
            name        = voice,
            orientation = 'h',
            x           = (durs[active].astype(float) / total_offsets).tolist(),
            y           = [voice] * int(active.sum()),
            base        = output.loc[active, 'Progress'].tolist(),
            marker      = dict(color=color, line=dict(width=0)),
            customdata  = pd.DataFrame({
                'measure': output.loc[active, 'Measure'].astype(int),
                'beat':    output.loc[active, 'Beat'],
                'pitch':   notes[active].astype(str),
                'dur':     durs[active].astype(float),
                'lyric':   lyrics,
            }).values,
            hovertemplate = (
                f'<b>{voice}</b><br>'
                'M %{customdata[0]}  b %{customdata[1]}<br>'
                'Pitch: %{customdata[2]}  Dur: %{customdata[3]:.0f}<br>'
                'Lyric: %{customdata[4]}<br>'
                '<extra></extra>'
            ),
        ), row=row, col=col)

fig = go.Figure()
add_voice_traces(fig)
fig.update_layout(
    title        = f"Voice activity: {output['Title'].iloc[0]}  —  {output['Composer'].iloc[0]}",
    barmode      = 'overlay',
    height       = 150 + len(voices) * 45,
    showlegend   = False,
    plot_bgcolor = 'white',
    margin       = dict(l=80, r=20, t=70, b=40),
    xaxis        = dict(range=[0, 1], showgrid=False, showticklabels=False),
    yaxis        = dict(title=None, categoryorder='array', categoryarray=list(reversed(voices))),
    shapes       = measure_lines + cad_lines,
    annotations  = measure_labels + cad_labels,
)
fig.show()
download_html(fig, f'voice_activity_{file_stem}.html')

### Sonorità complessiva

Ogni colonna indica quante voci sono attive in quel momento. L'asse x è lo stesso del grafico precedente; le cadenze sono riportate con le stesse linee rosse.

In [ ]:
sounding = (output[['Progress', 'Measure', 'Beat', 'Sounding']]
            .dropna(subset=['Sounding'])
            .astype({'Measure': int})
            .reset_index(drop=True))

pos    = sounding['Progress'].values
widths = np.append(np.diff(pos), 1.0 - pos[-1])

fig_s = go.Figure(go.Bar(
    x             = (pos + widths / 2).tolist(),
    y             = sounding['Sounding'].tolist(),
    width         = widths.tolist(),
    marker        = dict(color='#4472C4', line=dict(width=0)),
    customdata    = sounding[['Measure', 'Beat', 'Sounding']].values,
    hovertemplate = (
        'M %{customdata[0]}  b %{customdata[1]}<br>'
        'Sounding: %{customdata[2]:.0f}<br>'
        '<extra></extra>'
    ),
))
fig_s.update_layout(
    title        = f"Sounding voices: {output['Title'].iloc[0]}  —  {output['Composer'].iloc[0]}",
    height       = 250,
    showlegend   = False,
    plot_bgcolor = 'white',
    bargap       = 0,
    margin       = dict(l=60, r=20, t=70, b=40),
    xaxis        = dict(range=[0, 1], showgrid=False, showticklabels=False),
    yaxis        = dict(title='Sounding count', dtick=1),
    shapes       = measure_lines + cad_lines,
    annotations  = measure_labels + cad_labels,
)
fig_s.show()
download_html(fig_s, f'sounding_voices_{file_stem}.html')

### Vista combinata

Questo grafico offre una visione combinata dei due grafici precedenti.

In [ ]:
from plotly.subplots import make_subplots

fig_c = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.72, 0.28],
    vertical_spacing=0.04,
)
add_voice_traces(fig_c, row=1, col=1)

fig_c.add_trace(go.Bar(
    x             = (pos + widths / 2).tolist(),
    y             = sounding['Sounding'].tolist(),
    width         = widths.tolist(),
    marker        = dict(color='#4472C4', line=dict(width=0)),
    customdata    = sounding[['Measure', 'Beat', 'Sounding']].values,
    showlegend    = False,
    hovertemplate = (
        'M %{customdata[0]}  b %{customdata[1]}<br>'
        'Sounding: %{customdata[2]:.0f}<br>'
        '<extra></extra>'
    ),
), row=2, col=1)

sep_y    = (fig_c.layout.yaxis2.domain[1] + fig_c.layout.yaxis.domain[0]) / 2
sep_line = dict(type='line', x0=0, x1=1, y0=sep_y, y1=sep_y,
                xref='paper', yref='paper', line=dict(color='#aaa', width=1))

fig_c.update_layout(
    title        = f"{output['Title'].iloc[0]}  —  {output['Composer'].iloc[0]}",
    barmode      = 'overlay',
    height       = 150 + len(voices) * 45 + 180,
    showlegend   = False,
    plot_bgcolor = 'white',
    margin       = dict(l=80, r=20, t=70, b=40),
    shapes       = cad_lines + [sep_line],
    annotations  = measure_labels + cad_labels,
)
fig_c.update_xaxes(range=[0, 1], showgrid=False, showticklabels=False)
fig_c.update_yaxes(title_text=None, categoryorder='array',
                   categoryarray=list(reversed(voices)), row=1, col=1)
fig_c.update_yaxes(title_text='Sounding count', dtick=1, row=2, col=1)
fig_c.show()
download_html(fig_c, f'combined_{file_stem}.html')